In [ ]:
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score

ROOT   = "/content/drive/MyDrive/AI_Research/Model_Results/SaliencyRun/xrv_pipeline_nb"
LABELS = "/content/drive/MyDrive/AI_Research/stage_2_train_labels.csv"
REF    = f"{ROOT}/grad_vanilla_npy_25"
GC_CSV = f"{ROOT}/paper_outputs/T1_per_seed.csv"
SCORE  = 1024

seeds = sorted(int(d.split("_")[1]) for d in os.listdir(REF) if d.startswith("seed_"))
test_ids = sorted(f[:-4] for f in os.listdir(f"{REF}/seed_{seeds[0]}") if f.endswith(".npy"))
assert len(test_ids) == 601

lab = pd.read_csv(LABELS)
lab = lab[lab["Target"] == 1].dropna(subset=["x", "y", "width", "height"])
groups = dict(tuple(lab.groupby("patientId")))

def mask(rows):
    m = np.zeros((SCORE, SCORE), dtype=bool)
    for r in rows.itertuples():
        x0, y0 = int(round(r.x)), int(round(r.y))
        x1, y1 = int(round(r.x + r.width)), int(round(r.y + r.height))
        m[y0:y1, x0:x1] = True
    return m

train_ids = [p for p in groups if p not in set(test_ids)]
acc = np.zeros((SCORE, SCORE), dtype=np.float64)
for p in train_ids:
    acc += mask(groups[p])
avg_box = (acc / len(train_ids)).astype(np.float32)
np.save(f"{ROOT}/avg_box_map.npy", avg_box)
flat = avg_box.ravel()

aps = np.array([average_precision_score(mask(groups[p]).ravel(), flat) for p in test_ids])
base = float(aps.mean())

gc = pd.read_csv(GC_CSV).set_index("seed")["auprc_mean"].loc[seeds].values
vg_mean = 0.369

out = {
    "n_train_patients_in_baseline": len(train_ids),
    "avg_box_auprc_on_601": base,
    "gradcam_mean": float(gc.mean()),
    "gradcam_seeds_above_baseline": int((gc > base).sum()),
    "vanilla_mean": vg_mean,
    "vanilla_above_baseline": bool(vg_mean > base),
    "centre_prior_reference": 0.281,
}
print(json.dumps(out, indent=2))
json.dump(out, open(f"{ROOT}/avg_box_baseline.json", "w"), indent=2)